# Process weather data

In [5]:
from pathlib import Path
from solhycool_evaluation.utils import preprocess_meteonorm_txt_data
import pandas as pd

%load_ext autoreload
%autoreload 2

data_path: Path = Path("../../data")
meteo_data_file_path: Path = data_path / "datasets/partial/tmy_2005_Borg_El_Arab.txt"
output_path: Path = Path("../results/visualizations")

df_env = preprocess_meteonorm_txt_data(meteo_data_file_path)
df_env.head()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


2025-04-04 06:40:40.923 | INFO     | solhycool_evaluation.utils:preprocess_meteonorm_txt_data:42 - Pre-processed data saved to ../../data/datasets/partial/tmy_2005_Borg_El_Arab.csv and tmy_2005_Borg_El_Arab.h5


,G_Gh,G_Dh,G_Gk,G_Bn,Tamb_C,HR_pct,hs,Az,precip_mm,Td
time,,,,,,,,,,
2005-01-01 00:00:00+00:00,0,0,0,0,13.3,84,0.0,-142.6,0.5,10.7
2005-01-01 01:00:00+00:00,0,0,0,0,13.2,72,0.0,-107.2,0.0,8.2
2005-01-01 02:00:00+00:00,0,0,0,0,13.0,74,0.0,-94.4,0.0,8.4
2005-01-01 03:00:00+00:00,0,0,0,0,12.9,77,0.0,-86.5,0.0,8.9
2005-01-01 04:00:00+00:00,0,0,0,0,12.8,81,0.0,-80.1,0.0,9.7


In [3]:
# Visualize data
from plotly_resampler import FigureWidgetResampler
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from phd_visualizations.constants import generate_plotly_config

df_env = pd.read_hdf(meteo_data_file_path.with_suffix(".h5"))

var_ids: list[str] = ["Tamb_C", "HR_pct", "precip_mm"]
var_units: list[str] = ["°C", "%", "mm"]

fig = make_subplots(rows=len(var_ids), cols=1, shared_xaxes=True)
fig = FigureWidgetResampler(fig)

for i, (var_id, var_unit) in enumerate(zip(var_ids, var_units)):
    fig.add_trace(
        go.Scattergl(name=var_id, showlegend=True), 
        hf_x=df_env.index, 
        hf_y=df_env[var_id], 
        # max_n_samples=2_000,
        row=i + 1, col=1
    )
    fig.update_yaxes(title_text=f"{var_id.replace("_", " ")} ({var_unit})", row=i + 1)

fig.update_layout(
    height=650,
    width=800,
    title="<b>Weather variables</b>",
    title_x=0.1,
    legend_traceorder="normal",
    legend=dict(orientation="h", y=1.08, xanchor="left", x=0),
)

fig

# fig.show(
#     config=generate_plotly_config(
#         fig, figure_name=f"solhycool_env_viz_{df_env.index[0].strftime('%Y%m%d')}"
#     )
# )



FigureWidgetResampler({
    'data': [{'name': '<b style="color:sandybrown">[R]</b> Tamb_C <i style="color:#fc9944">~9h</i>',
              'showlegend': True,
              'type': 'scattergl',
              'uid': 'cd0ffc5c-555e-4083-9ba3-49d80b415e24',
              'x': array([datetime.datetime(2005, 1, 1, 0, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2005, 1, 1, 6, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2005, 1, 1, 13, 0, tzinfo=datetime.timezone.utc), ...,
                          datetime.datetime(2005, 12, 31, 6, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2005, 12, 31, 14, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2005, 12, 31, 23, 0, tzinfo=datetime.timezone.utc)],
                         shape=(1000,), dtype=object),
              'xaxis': 'x',
              'y': array([13.3, 12.7, 21.4, ..., 10.1, 17.1,  8.2], shape=(1000,)),
     

In [6]:
from phd_visualizations import save_figure

start, end = fig.layout.xaxis.range
save_figure(fig, f"{meteo_data_file_path.stem}_viz_{start[:10].replace('-', '')}_{end[:10].replace('-', '')}", 
            figure_path=output_path, 
            formats=["png", "svg"])


2025-04-04 06:40:55.064 | INFO     | phd_visualizations:save_figure:38 - Figure saved in [PosixPath('../results/visualizations')]/tmy_2005_Borg_El_Arab_viz_20050101_20051231.png
2025-04-04 06:40:57.270 | INFO     | phd_visualizations:save_figure:38 - Figure saved in [PosixPath('../results/visualizations')]/tmy_2005_Borg_El_Arab_viz_20050101_20051231.svg
